### Planetscope / RapidEye preprocessing - Reflectance Mosaic (4 band and 8 band PS)

by Miguel Angel Ramírez

21.01.2024

This script:
- Reads multiple Planetscope and RapidEye images/scenes from .zip archive
- Convert TOA Radiance to TOA Reflectance with application of cloud mask and compression
- Creates a composite image (or mosaic) 

Code is based on notebooks from Planetlabs, available here: https://github.com/planetlabs/notebooks

It improves the using of virtual memory

Data used are from Planetscope January of 2021 over a selected study area in Montes de Maria.

The results are PlanetScope reflectance orthoimages and the mosaic

***

In [1]:
import numpy as np
import matplotlib
import rasterio
import fiona
import os
import zipfile
import shutil
import cv2
import gdal
import shapely
import itertools

from rasterio.enums import Resampling
from osgeo import gdal
from os import walk
from rasterio.merge import merge

### Functions

In [2]:
def aplicar_filtro_nubosidad(input_path, udm2_path, output_path, cat):
    # Abrir la imagen satelital PlanetScope y el archivo UDM2
    with rasterio.open(input_path) as src, rasterio.open(udm2_path) as udm2:
        # Obtener información sobre la imagen de entrada
        profile = src.profile

        # Crear un archivo de salida con el mismo perfil que la imagen de entrada
        profile.update(dtype=np.uint16, compress='lzw')  # Configurar el tipo de datos y la compresión

        with rasterio.open(output_path, 'w', **profile) as dst:
            for band in range(1, src.count + 1):
                # Leer la banda actual de la imagen satelital
                image_band = src.read(band)

                # Leer la primera banda de UDM2 (la máscara)
                mask_band = udm2.read(1)
                
                if cat == 1:
                    # Invertir la máscara: asignar 1 a los valores cero y 0 a los valores mayores que cero
                    mask_band = (mask_band == 0).astype(np.uint8)

                # Aplicar el filtro de nubosidad a la banda actual
                filtered_band = image_band * mask_band

                # Convertir la banda filtrada a uint16 para coincidir con el tipo de datos del archivo de salida
                filtered_band = filtered_band.astype(np.uint16)

                # Escribir la banda filtrada en el archivo de salida
                dst.write(filtered_band, band)

def compress_and_mask_image(input_file, output_file, compression, cat):
    
    if "AnalyticMS" in input_file:
        mask_file = Project_folder + '/Mask/' + input_file.split('/')[-1][:26] + '_udm2.tif'
    else:
        mask_file = Project_folder + '/Mask/' + input_file.split('/')[-1][:31] + '_udm2.tif'

    # Validar los parámetros de entrada
    if not os.path.exists(input_file):
        raise ValueError('El archivo de entrada no existe')

    # Validar la existencia del archivo de máscara
    if not os.path.exists(mask_file):
        mask_file = Project_folder + '/Mask/' + input_file.split('/')[-1][:37] + 'DN_udm.tif'
        if not os.path.exists(mask_file):
            mask_file = Project_folder + '/Mask/' + input_file.split('/')[-1][:36] + 'DN_udm.tif'
            if not os.path.exists(mask_file):
                mask_file = Project_folder + '/Mask/' + input_file.split('/')[-1][:25] + '_udm.tif'
                if not os.path.exists(mask_file):
                    raise ValueError('El archivo de máscara no existe')

    # Aplicar la máscara
    aplicar_filtro_nubosidad(input_file, mask_file, output_file, cat)
    

def unzip_and_move_files(zip_file, output_folder, udm, PSt):
    """Unzip a ZIP file and move the files to the specified folder."""

    # Unzip the file
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(output_folder)

    # Get the PSScene folder
    if PSt == 1:
        d = "files"
    else:
        d = ""

    possible_folders = ["PSScene", "REOrthoTile", "PSOrthoTile"]
    PSScene_folder = None

    for folder in possible_folders:
        folder_path = os.path.join(output_folder, d, folder)
        if os.path.isdir(folder_path):
            PSScene_folder = folder_path
            break

    if PSScene_folder is None:
        raise FileNotFoundError("No PSScene, REOrthoTile, or PSOrthoTile folder found.")
    # Create the folders
    os.makedirs(os.path.join(output_folder, 'Mask'), exist_ok=True)
    os.makedirs(os.path.join(output_folder, 'AnalyticMS_files'), exist_ok=True)
    os.makedirs(os.path.join(output_folder, 'Metadata_files'), exist_ok=True)
    os.makedirs(os.path.join(output_folder, 'Results'), exist_ok=True)

    # Move the files
    for file in os.listdir(PSScene_folder):
        file_path = os.path.join(PSScene_folder, file)

        if PSt == 1:
            if os.path.isdir(file_path):
                file = os.path.join(file_path, "analytic_sr")

            for infile in os.listdir(file):
                infile_path = os.path.join(file, infile)
                if infile.endswith('.xml'):
                    shutil.move(infile_path, os.path.join(output_folder, 'Metadata_files'))
                elif infile.endswith('.tif'):
                    if udm == 'udm':
                        if 'udm' in infile:
                            shutil.move(infile_path, os.path.join(output_folder, 'Mask'))
                        else:
                            shutil.move(infile_path, os.path.join(output_folder, 'AnalyticMS_files'))
                    else:
                        shutil.move(infile_path, os.path.join(output_folder, 'AnalyticMS_files'))
        else:
            if file.endswith('.tif'):
                # Check if the file is a mask
                if (udm == 'udm2'or udm == "udm") and ('udm2' in file or '_udm' in file):
                    shutil.move(file_path, os.path.join(output_folder, 'Mask'))
                else:
                    shutil.move(file_path, os.path.join(output_folder, 'AnalyticMS_files'))

    # Move the metadata file
    for file in os.listdir(PSScene_folder):
        file_path = os.path.join(PSScene_folder, file)
        if file.endswith('.xml'):
            shutil.move(file_path, os.path.join(output_folder, 'Metadata_files'))

    # Delete the unzipped folder
    shutil.rmtree(PSScene_folder)
    if PSt == 1:
        shutil.rmtree(os.path.join(output_folder, d))

def scaled_export(images, nbands, dgh):
    
    if (nbands==8):
    
        costblue_ref_scaled = [0]*n_images
        blue_ref_scaled = [0]*n_images
        greenI_ref_scaled = [0]*n_images
        greenII_ref_scaled = [0]*n_images
        yellow_ref_scaled = [0]*n_images
        red_ref_scaled = [0]*n_images
        rededge_ref_scaled = [0]*n_images
        nir_ref_scaled = [0]*n_images


        for i in range(len(images)):
            # Get the metadata of original GeoTIFF:
            meta = images[i].meta
            # Set the source metadata as kwargs we'll use to write the new data:
            # update the 'dtype' value to uint16:
            kwargs = meta
            kwargs.update(dtype='uint16')

            # As noted above, scale reflectance value by a factor of 10k:
            scale = 10000
            costblue_ref_scaled[i] = scale * band_costblue_reflectance[0]
            del band_costblue_reflectance[0]
            blue_ref_scaled[i] = scale * band_blue_reflectance[0]
            del band_blue_reflectance[0]
            greenI_ref_scaled[i] = scale * band_greenI_reflectance[0]
            del band_greenI_reflectance[0]
            greenII_ref_scaled[i] = scale * band_greenII_reflectance[0]
            del band_greenII_reflectance[0]
            yellow_ref_scaled[i] = scale * band_yellow_reflectance[0]
            del band_yellow_reflectance[0]
            red_ref_scaled[i] = scale * band_red_reflectance[0]
            del band_red_reflectance[0]
            rededge_ref_scaled [i] = scale * band_redegde_reflectance[0]
            del band_redegde_reflectance[0]
            nir_ref_scaled[i] = scale * band_nir_reflectance[0]
            del band_nir_reflectance[0]

            # Convert to type 'uint16'
            from rasterio import uint16
            costblue_scaled = costblue_ref_scaled[i].astype(uint16)
            blue_scaled = blue_ref_scaled[i].astype(uint16)
            greenI_scaled = greenI_ref_scaled[i].astype(uint16)
            greenII_scaled = greenII_ref_scaled[i].astype(uint16)
            yellow_scaled = yellow_ref_scaled[i].astype(uint16)
            red_scaled = red_ref_scaled[i].astype(uint16)
            rededgde_scaled = rededge_ref_scaled[i].astype(uint16)
            nir_scaled = nir_ref_scaled[i].astype(uint16)

            # New name for exported image
            img_name_ref_scaled = (Results_folder + AnalyticMS_files[i]).replace('.tif','_Ref.tif')

            # Write band calculations to a new unscaled GeoTIFF file with same band order (BGRN)
            with rasterio.open(img_name_ref_scaled, 'w', **kwargs) as dst:
                    dst.write_band(1, costblue_scaled)
                    del costblue_scaled
                    dst.write_band(2, blue_scaled)
                    del blue_scaled
                    dst.write_band(3, greenI_scaled)
                    del greenI_scaled
                    dst.write_band(4, greenII_scaled)
                    del greenII_scaled
                    dst.write_band(5, yellow_scaled)
                    del yellow_scaled
                    dst.write_band(6, red_scaled)
                    del red_scaled
                    dst.write_band(7, rededgde_scaled)
                    del rededgde_scaled
                    dst.write_band(8, nir_scaled)
                    del nir_scaled

            output_file = img_name_ref_scaled.replace('_Ref.tif','_Ref_comp.tif')
            compress_and_mask_image(img_name_ref_scaled, output_file, "LZW", dgh)
            os.remove(img_name_ref_scaled)

        del costblue_ref_scaled
        del blue_ref_scaled
        del greenI_ref_scaled
        del greenII_ref_scaled
        del yellow_ref_scaled
        del red_ref_scaled
        del rededge_ref_scaled
        del nir_ref_scaled
    elif (nbands==5):
        blue_ref_scaled = [0]*n_images
        greenI_ref_scaled = [0]*n_images
        red_ref_scaled = [0]*n_images
        rededge_ref_scaled = [0]*n_images
        nir_ref_scaled = [0]*n_images

        for i in range(len(images)):
            # Get the metadata of original GeoTIFF:
            meta = images[i].meta
            # Set the source metadata as kwargs we'll use to write the new data:
            # update the 'dtype' value to uint16:
            kwargs = meta
            kwargs.update(dtype='uint16')

            # As noted above, scale reflectance value by a factor of 10k:
            scale = 1
            blue_ref_scaled[i] = scale * band_blue_reflectance[0]
            del band_blue_reflectance[0]
            greenI_ref_scaled[i] = scale * band_greenI_reflectance[0]
            del band_greenI_reflectance[0]
            red_ref_scaled[i] = scale * band_red_reflectance[0]
            del band_red_reflectance[0]
            rededge_ref_scaled [i] = scale * band_redegde_reflectance[0]
            del band_redegde_reflectance[0]
            nir_ref_scaled[i] = scale * band_nir_reflectance[0]
            del band_nir_reflectance[0]

            # Convert to type 'uint16'
            from rasterio import uint16
            blue_scaled = blue_ref_scaled[i].astype(uint16)
            greenI_scaled = greenI_ref_scaled[i].astype(uint16)
            red_scaled = red_ref_scaled[i].astype(uint16)
            rededgde_scaled = rededge_ref_scaled[i].astype(uint16)
            nir_scaled = nir_ref_scaled[i].astype(uint16)

            # New name for exported image
            img_name_ref_scaled = (Results_folder + AnalyticMS_files[i]).replace('.tif','_Ref.tif')

            # Write band calculations to a new unscaled GeoTIFF file with same band order (BGRN)
            with rasterio.open(img_name_ref_scaled, 'w', **kwargs) as dst:
                    dst.write_band(1, blue_scaled)
                    del blue_scaled
                    dst.write_band(2, greenI_scaled)
                    del greenI_scaled
                    dst.write_band(3, red_scaled)
                    del red_scaled
                    dst.write_band(4, rededgde_scaled)
                    del rededgde_scaled
                    dst.write_band(5, nir_scaled)
                    del nir_scaled

            output_file = img_name_ref_scaled.replace('_Ref.tif','_Ref_comp.tif')
            compress_and_mask_image(img_name_ref_scaled, output_file, "LZW", dgh)
            os.remove(img_name_ref_scaled)
            
        del blue_ref_scaled
        del greenI_ref_scaled
        del red_ref_scaled
        del rededge_ref_scaled
        del nir_ref_scaled
        
        
    else:
        blue_ref_scaled = [0]*n_images
        greenI_ref_scaled = [0]*n_images
        red_ref_scaled = [0]*n_images
        nir_ref_scaled = [0]*n_images


        for i in range(len(images)):
            # Get the metadata of original GeoTIFF:
            meta = images[i].meta
            # Set the source metadata as kwargs we'll use to write the new data:
            # update the 'dtype' value to uint16:
            kwargs = meta
            kwargs.update(dtype='uint16')

            # As noted above, scale reflectance value by a factor of 10k:
            scale = 10000
            blue_ref_scaled[i] = scale * band_blue_reflectance[0]
            del band_blue_reflectance[0]
            greenI_ref_scaled[i] = scale * band_greenI_reflectance[0]
            del band_greenI_reflectance[0]
            red_ref_scaled[i] = scale * band_red_reflectance[0]
            del band_red_reflectance[0]
            nir_ref_scaled[i] = scale * band_nir_reflectance[0]
            del band_nir_reflectance[0]

            # Convert to type 'uint16'
            from rasterio import uint16
            blue_scaled = blue_ref_scaled[i].astype(uint16)
            greenI_scaled = greenI_ref_scaled[i].astype(uint16)
            red_scaled = red_ref_scaled[i].astype(uint16)
            nir_scaled = nir_ref_scaled[i].astype(uint16)

            # New name for exported image
            img_name_ref_scaled = (Results_folder + AnalyticMS_files[i]).replace('.tif','_Ref.tif')

            # Write band calculations to a new unscaled GeoTIFF file with same band order (BGRN)
            with rasterio.open(img_name_ref_scaled, 'w', **kwargs) as dst:
                    dst.write_band(1, blue_scaled)
                    del blue_scaled
                    dst.write_band(2, greenI_scaled)
                    del greenI_scaled
                    dst.write_band(3, red_scaled)
                    del red_scaled
                    dst.write_band(4, nir_scaled)
                    del nir_scaled

            output_file = img_name_ref_scaled.replace('_Ref.tif','_Ref_comp.tif')
            compress_and_mask_image(img_name_ref_scaled, output_file, "LZW", dgh)
            os.remove(img_name_ref_scaled)
            
        del blue_ref_scaled
        del greenI_ref_scaled
        del red_ref_scaled
        del nir_ref_scaled


def delete_folders(input_dir):
    
    folders = ['AnalyticMS_files', 'Mask', 'Metadata_files']
    for folder in folders:
        folder_path = os.path.join(input_dir, folder)
        files = os.listdir(folder_path)
        for file in files:
            file_path = os.path.join(folder_path, file)
            os.remove(file_path)
        os.remove(folder_path)

# Function to export the unclipped scaled mosaic
def mosaic_scaled_export(images, name):
        # Get the metadata of original GeoTIFF:
        (mosaic_scaled, transform) = merge(img_list_scaled)
        meta = images[0].meta
        
        # Update the original metadata to reflect the specifics of our new mosaic
        meta.update({"transform": transform,
                     "height":mosaic_scaled.shape[1],
                     "width":mosaic_scaled.shape[2]})
        
        # New name for exported scaled mosaic
        os.makedirs(os.path.join(Project_folder, 'Mosaic'), exist_ok=True)
        img_name_mosaic_scaled = (Project_folder + "/Mosaic/" + name + '.tif')
        
        # Write band calculations to a new scaled GeoTIFF file
        with rasterio.open(img_name_mosaic_scaled, 'w', **meta) as dst:
                dst.write(mosaic_scaled)

### Step 0 Unzip the file of PlanetScope Scenes

It is vital to define the Project folder and the route of .zip which have the PlanetScope orthoimages

In [3]:
# Unzip the file
Project_folder = "E:\TOLIMA 2011"
unzip_and_move_files(r'D:\DESCARGAS\montes_2011y_reorthotile_analytic_sr.zip', Project_folder, "udm", 0)

#### Step 1. Read and Inspect Images

In [4]:
AnalyticMS_folder = Project_folder + '/AnalyticMS_files/'
Metadata_folder = Project_folder +'/Metadata_files/'
Results_folder = Project_folder + '/Results/'

AnalyticMS_files = []
for (dirpath, dirnames, filenames) in walk(AnalyticMS_folder):
    AnalyticMS_files.extend(filenames)
    break

# Add image file path
AnalyticMS_path = []
for i in AnalyticMS_files:
    AnalyticMS_path.append(AnalyticMS_folder + i)
    print(AnalyticMS_folder + i) # Verify correct images

# Loading images into list using rasterio
img_list = []
for image in AnalyticMS_path:
    img_list.append(rasterio.open(image))
n_images = len(img_list)

E:\TOLIMA 2011/AnalyticMS_files/1843414_2011-01-14_RE2_3A_Analytic_SR.tif
E:\TOLIMA 2011/AnalyticMS_files/1843414_2011-01-30_RE4_3A_Analytic_SR.tif
E:\TOLIMA 2011/AnalyticMS_files/1843414_2011-02-04_RE4_3A_Analytic_SR.tif
E:\TOLIMA 2011/AnalyticMS_files/1843415_2011-01-30_RE4_3A_Analytic_SR.tif
E:\TOLIMA 2011/AnalyticMS_files/1843415_2011-02-04_RE4_3A_Analytic_SR.tif
E:\TOLIMA 2011/AnalyticMS_files/1843416_2011-01-30_RE4_3A_Analytic_SR.tif
E:\TOLIMA 2011/AnalyticMS_files/1843416_2011-02-04_RE4_3A_Analytic_SR.tif
E:\TOLIMA 2011/AnalyticMS_files/1843513_2011-01-14_RE2_3A_Analytic_SR.tif
E:\TOLIMA 2011/AnalyticMS_files/1843513_2011-02-04_RE4_3A_Analytic_SR.tif
E:\TOLIMA 2011/AnalyticMS_files/1843514_2011-01-14_RE2_3A_Analytic_SR.tif
E:\TOLIMA 2011/AnalyticMS_files/1843514_2011-01-30_RE4_3A_Analytic_SR.tif
E:\TOLIMA 2011/AnalyticMS_files/1843514_2011-02-04_RE4_3A_Analytic_SR.tif
E:\TOLIMA 2011/AnalyticMS_files/1843515_2011-01-14_RE2_3A_Analytic_SR.tif
E:\TOLIMA 2011/AnalyticMS_files/184351

In [5]:
# Reading image metadata
Analytic_meta_files = []
for (dirpath, dirnames, filenames) in os.walk(Metadata_folder):
    Analytic_meta_files.extend(filenames)
    break

# Add image metadata file path
Analytic_meta_path = []
for i in Analytic_meta_files:
    Analytic_meta_path.append(Metadata_folder + i)
    print(Metadata_folder + i) # Verify correct metadata

E:\TOLIMA 2011/Metadata_files/1843414_2011-01-14_RE2_3A_Analytic_metadata.xml
E:\TOLIMA 2011/Metadata_files/1843414_2011-01-30_RE4_3A_Analytic_metadata.xml
E:\TOLIMA 2011/Metadata_files/1843414_2011-02-04_RE4_3A_Analytic_metadata.xml
E:\TOLIMA 2011/Metadata_files/1843415_2011-01-30_RE4_3A_Analytic_metadata.xml
E:\TOLIMA 2011/Metadata_files/1843415_2011-02-04_RE4_3A_Analytic_metadata.xml
E:\TOLIMA 2011/Metadata_files/1843416_2011-01-30_RE4_3A_Analytic_metadata.xml
E:\TOLIMA 2011/Metadata_files/1843416_2011-02-04_RE4_3A_Analytic_metadata.xml
E:\TOLIMA 2011/Metadata_files/1843513_2011-01-14_RE2_3A_Analytic_metadata.xml
E:\TOLIMA 2011/Metadata_files/1843513_2011-02-04_RE4_3A_Analytic_metadata.xml
E:\TOLIMA 2011/Metadata_files/1843514_2011-01-14_RE2_3A_Analytic_metadata.xml
E:\TOLIMA 2011/Metadata_files/1843514_2011-01-30_RE4_3A_Analytic_metadata.xml
E:\TOLIMA 2011/Metadata_files/1843514_2011-02-04_RE4_3A_Analytic_metadata.xml
E:\TOLIMA 2011/Metadata_files/1843515_2011-01-14_RE2_3A_Analytic

In [6]:
# Inspecting image metadata
print("Image: dtype | crs | band count")
for image in img_list:
    nband = image.meta['count']
    print(image.meta['dtype'], image.meta['crs'], image.meta['count'])

Image: dtype | crs | band count
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5
uint16 EPSG:32618 5


## Step 2. Extract the Data from Each Spectral Band
In this step, Rasterio (a Python library for reading and writing geospatial raster datasets) is used to open the raster images (the .tif files). 

Then, the band data will is extracted and loaded into arrays for further manipulation with Python's NumPy libary.

Note: in PlanetScope 8-band images, the band order is: (1) Coastal Blue, (2) Blue, (3) Green I, (4) Green II, (5) Yellow, (6) Red, (7) Red Edge, (8) Near-infrared.

in PlanetScope 4-band images, the band order is: (1) Blue, (2) Green, (3) Red, (4) Near-infrared.

In [7]:
if (nband == 8):
    # Radiance values are loaded into lists
    band_costblue_radiance = [0]*n_images
    band_blue_radiance = [0]*n_images
    band_greenI_radiance = [0]*n_images
    band_greenII_radiance = [0]*n_images
    band_yellow_radiance = [0]*n_images
    band_red_radiance = [0]*n_images
    band_rededge_radiance = [0]*n_images
    band_nir_radiance = [0]*n_images
    
    # Using rasterio to read radiance values for the images
    for i in range(n_images):
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_costblue_radiance[i] = src.read(1)
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_blue_radiance[i] = src.read(2)
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_greenI_radiance[i] = src.read(3)
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_greenII_radiance[i] = src.read(4)
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_yellow_radiance[i] = src.read(5)
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_red_radiance[i] = src.read(6)
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_rededge_radiance[i] = src.read(7)
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_nir_radiance[i] = src.read(8)
elif (nband == 5):
    # Radiance values are loaded into lists
    band_blue_radiance = [0]*n_images
    band_greenI_radiance = [0]*n_images
    band_red_radiance = [0]*n_images
    band_rededge_radiance = [0]*n_images
    band_nir_radiance = [0]*n_images
    
    # Using rasterio to read radiance values for the images
    for i in range(n_images):
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_blue_radiance[i] = src.read(1)
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_greenI_radiance[i] = src.read(2)
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_red_radiance[i] = src.read(3)
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_rededge_radiance[i] = src.read(4)
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_nir_radiance[i] = src.read(5)
else:
    # Radiance values are loaded into lists
    band_blue_radiance = [0]*n_images
    band_greenI_radiance = [0]*n_images
    band_red_radiance = [0]*n_images
    band_nir_radiance = [0]*n_images
    
    # Using rasterio to read radiance values for the images
    for i in range(n_images):
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_blue_radiance[i] = src.read(1)
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_greenI_radiance[i] = src.read(2)
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_red_radiance[i] = src.read(3)
        with rasterio.open(AnalyticMS_path[i]) as src:
            band_nir_radiance[i] = src.read(4)

In [8]:
print(band_blue_radiance)

[array([[ 182,  206,  241, ...,   39,    9,    7],
       [ 223,  277,  303, ...,    7,    6,    7],
       [ 179,  224,  254, ...,    5,    6,    9],
       ...,
       [2437, 2320, 2258, ..., 6542, 6448, 6372],
       [2784, 2748, 2732, ..., 6384, 6248, 6186],
       [3289, 3308, 3192, ..., 6288, 6179, 6085]], dtype=uint16), array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 1, 1, 1]], dtype=uint16), array([[   3,    3,    3, ...,    0,    0,    0],
       [   3,    4,    4, ...,    0,    0,    0],
       [   5,    5,    4, ...,    0,    0,    0],
       ...,
       [4874, 4801, 4743, ...,    0,    0,    0],
       [4955, 4872, 4758, ...,    0,    0,    0],
       [5076, 4944, 4846, ...,    0,    0,    0]], dtype=uint16), array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [1, 1, 1

In [8]:
from xml.dom import minidom

coeffs_list = [] 

for image in Analytic_meta_path:
    xmldoc = minidom.parse(image)
    if nband ==5:
         nodes = xmldoc.getElementsByTagName("re:bandSpecificMetadata")
    else:
        nodes = xmldoc.getElementsByTagName("ps:bandSpecificMetadata")
        
    # XML parser refers to bands by numbers 1-4
    coeffs = {}  # Coefficients for each image/scene

    if nband == 8:
        arreg = ['1', '2', '3', '4', '5', '6', '7', '8']
        for node in nodes:
            band_nr = node.getElementsByTagName("ps:bandNumber")[0].firstChild.data
            if band_nr in arreg:
                i = int(band_nr)
                value = node.getElementsByTagName("ps:reflectanceCoefficient")[0].firstChild.data
                coeffs[i] = float(value)
    elif nband == 5:
        arreg = ['1', '2', '3', '4', '5']
        for band_nr in arreg:
            i = int(band_nr)
            #value = nodes[0].getElementsByTagName("re:radiometricScaleFactor")[0].firstChild.data
            value = 1
            coeffs[i] = float(value)
    else:
        arreg = ['1', '2', '3', '4']
        for node in nodes:
            band_nr = node.getElementsByTagName("ps:bandNumber")[0].firstChild.data
            if band_nr in arreg:
                i = int(band_nr)
                value = node.getElementsByTagName("ps:reflectanceCoefficient")[0].firstChild.data
                coeffs[i] = float(value)

    coeffs_list.append(coeffs)

for coffee in coeffs_list:
    print(coffee)  # Verify data

{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}
{1: 1.0, 2: 1.0,

#### Step 4: Convert Radiance to Reflectance
Radiance is measured in SI units: $W/m^2$. Reflectance is a ratio from 0 to 1. The conversion is performed as a per-band scalar multiplication:

In [9]:
if (nband == 8):
    # Radiance values are loaded into lists
    band_costblue_reflectance = [];   costblue_ref_scaled = [0]*n_images
    band_blue_reflectance = [];   blue_ref_scaled = [0]*n_images
    band_greenI_reflectance = [];  greenI_ref_scaled = [0]*n_images
    band_greenII_reflectance = [];  greenII_ref_scaled = [0]*n_images
    band_yellow_reflectance = [];    yellow_ref_scaled = [0]*n_images
    band_red_reflectance = [];    red_ref_scaled = [0]*n_images
    band_redegde_reflectance = [];    rededge_ref_scaled = [0]*n_images
    band_nir_reflectance = [];    nir_ref_scaled = [0]*n_images

    # Calculating reflectance from radiance and conversion coefficients
    for i in range(n_images):
        band_costblue_reflectance.append(band_costblue_radiance[i] * coeffs_list[i][1])
        band_blue_reflectance.append(band_blue_radiance[i] * coeffs_list[i][2])
        band_greenI_reflectance.append(band_greenI_radiance[i] * coeffs_list[i][3])
        band_greenII_reflectance.append(band_greenII_radiance[i] * coeffs_list[i][4])
        band_yellow_reflectance.append(band_yellow_radiance[i] * coeffs_list[i][5])
        band_red_reflectance.append(band_red_radiance[i] * coeffs_list[i][6])
        band_redegde_reflectance.append(band_rededge_radiance[i] * coeffs_list[i][7])
        band_nir_reflectance.append(band_nir_radiance[i] * coeffs_list[i][8])

    del band_costblue_radiance
    del band_blue_radiance
    del band_greenI_radiance
    del band_greenII_radiance
    del band_yellow_radiance
    del band_red_radiance
    del band_rededge_radiance
    del band_nir_radiance
    
if (nband == 5):
    # Radiance values are loaded into lists
    band_blue_reflectance = [];   blue_ref_scaled = [0]*n_images
    band_greenI_reflectance = [];  greenI_ref_scaled = [0]*n_images
    band_red_reflectance = [];    red_ref_scaled = [0]*n_images
    band_redegde_reflectance = [];    rededge_ref_scaled = [0]*n_images
    band_nir_reflectance = [];    nir_ref_scaled = [0]*n_images

    # Calculating reflectance from radiance and conversion coefficients
    for i in range(n_images):
        band_blue_reflectance.append(band_blue_radiance[i] * coeffs_list[i][1])
        band_greenI_reflectance.append(band_greenI_radiance[i] * coeffs_list[i][2])
        band_red_reflectance.append(band_red_radiance[i] * coeffs_list[i][3])
        band_redegde_reflectance.append(band_rededge_radiance[i] * coeffs_list[i][4])
        band_nir_reflectance.append(band_nir_radiance[i] * coeffs_list[i][5])

    del band_blue_radiance
    del band_greenI_radiance
    del band_red_radiance
    del band_rededge_radiance
    del band_nir_radiance
else:
    # Radiance values are loaded into lists
    band_blue_reflectance = [];   blue_ref_scaled = [0]*n_images
    band_greenI_reflectance = [];  greenI_ref_scaled = [0]*n_images
    band_red_reflectance = [];    red_ref_scaled = [0]*n_images
    band_nir_reflectance = [];    nir_ref_scaled = [0]*n_images

    # Calculating reflectance from radiance and conversion coefficients
    for i in range(n_images):
        band_blue_reflectance.append(band_blue_radiance[i] * coeffs_list[i][1])
        band_greenI_reflectance.append(band_greenI_radiance[i] * coeffs_list[i][2])
        band_red_reflectance.append(band_red_radiance[i] * coeffs_list[i][3])
        band_nir_reflectance.append(band_nir_radiance[i] * coeffs_list[i][4])

    del band_blue_radiance
    del band_greenI_radiance
    del band_red_radiance
    del band_nir_radiance

#### Step 5. Write the Reflectance Images
A note: Reflectance is generally defined as a floating point number between 0 and 1, but image file formats are much more commonly stored as unsigned integers. A common practice in the industry is to multiply the reflectance value by a *scale factor* of 10,000, then save the result as a file with data type uint16.


Export of normal scaled GeoTIFF (multiplied with scale factor of 10 000 and stored as *dtype=uint16*)

##### Export Reflectance Images to your project folder

The images reflectance output have the cloud mask applied and compression (LZW) (First result)

In [10]:
# Export scaled images Reflectance:
scaled_export(img_list, nband, 1)

In [11]:
### Load image results into lists for the clip functions
Results_files = []
Results_path_scaled = []; Results_path_unscaled = []
img_list_scaled = []; img_list_unscaled = []

for (dirpath, dirnames, filenames) in walk(Results_folder):
    Results_files.extend(filenames)
    break

for filename in Results_files:
    if "Ref_comp.tif" in filename:
        Results_path_scaled.append(Results_folder + filename)
        img_list_scaled.append(rasterio.open(Results_folder + filename))

#### Step 6. Creating the Mosaic and delete initial images

Export of scaled unclipped mosaic GeoTIFF (multiplied with scale factor of 10 000 and stored as *dtype=uint16*) compressed with LZW algorithm. It can view in the carpet of "Mosaic" (Second result)

The function has two arguments, the first one is the list of reflectance images and the name of Mosaic

In [12]:
# Export and calculated scaled mosaic:
mosaic_scaled_export(img_list, "Montes 2011")

In [ ]:
delete_folders(Project_folder)